In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import itertools
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from HelperFunctions import read_lesion_monster_file
from pathlib import Path
%load_ext rpy2.ipython

Read in data, create dataframe, set plotting parameters

In [ ]:
sns.set_theme(style="ticks", context="talk", font="Arial")

plt.rcParams.update({
    # Font + text sizes
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],  

    # Line widths
    "axes.linewidth": 0.5,   
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 2,     
    "ytick.major.size": 2,


    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42, 
    "text.usetex": False,
})


def sem(x):
    n = x.count()
    return x.std(ddof=1) / np.sqrt(n) if n > 1 else np.nan

In [ ]:
root = Path(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Lesion\Lesion')

fullpaths = []

for top in root.iterdir():
    if not top.is_dir():
        continue
    base = top.name
    for leaf in ("Reward3_1","Reward3_2","Reward3_3", "Reward3_4", "Reward3_5"):
        sub = top / leaf
        if not sub.is_dir():
            continue
        prefix = f"{base}_{leaf}".lower()

        # use sub.rglob('*') if files can be nested
        for f in sub.iterdir():
            if not f.is_file():
                continue
            name_l = f.name.lower()
            if name_l.startswith(prefix):
                # ensure Control_1 doesn't match Control_10
                n = len(prefix)
                if len(name_l) == n or name_l[n:n+1] in {'_', '-', '.'}:
                    fullpaths.append(str(f))
fullpaths = [p for p in fullpaths
             if not p.lower().endswith(('.csv', '.pickle', '.mp4', '.h5'))]

vsl = ("Gauguin", "Warhol", "Renoir", "Fragonard", "Turner", "Pollock", "Klimt", "Rembrandt", "Nessarose", "Morrible", "Boq", "Dorothy")

parts = []
for targetfile in fullpaths:
    animal = targetfile.split('\\')[-3]
    session = targetfile.split('\\')[-2]
    try:
        file = read_lesion_monster_file(targetfile, animal, session)

        #df =  pd.DataFrame(file[0]['T_beam'][:,0], columns = ['time'])
        df =  pd.DataFrame(file[1]['t_beam'], columns = ['time'])
        df['events'] = file[0]['T_beam'][:,1]

        door = pd.DataFrame(file[1]['t_door'], columns=['time'])
        door['events'] = 'Door'
        entrance = pd.DataFrame(file[1]['t_enter'], columns=['time'])
        entrance['events'] = 'Entrance'
        reward = pd.DataFrame(file[1]['t_reward'], columns=['time'])
        reward['events'] = 'Reward'
        monster = pd.DataFrame(file[1]['t_monster'], columns=['time'])
        monster['events'] = 'Monster'
        exit = pd.DataFrame(file[1]['t_exit'], columns=['time'])
        exit['events'] = 'Exit'

        df = pd.concat([df, door, entrance, reward, monster, exit], ignore_index=True) 
        df = df.sort_values('time').reset_index(drop=True)

        m = df['events'].astype(str).str.strip().str.casefold().eq('door')
        df['trial'] = m.cumsum()
        df['time'] = (df['time'] - df['time'][0])/1000

        df['animal'] = animal
        df['session'] = session
        if animal in vsl:
            df['lesion'] = 'Lesion'
        else:
            df['lesion'] = 'Saline'
        
        parts.append(df)

    except:
        print(targetfile)
mdf = pd.concat(parts)
mdf = mdf.sort_values(by=['animal', 'session', 'time'])
mdf['session_type'] = np.where(mdf['session'].str.contains('monster', case=False),'Monster','Reward')

mdf['session_n'] = mdf['session'].astype(str).str[-1].astype(int)

mdf['rewarded'] = (
    mdf.groupby(['animal', 'session', 'trial'])['events']
       .transform(lambda s: s.astype('string').str.contains('Reward', case=False, na=False).any())
       .map({True: 1, False: 0})
)

num = pd.to_numeric(mdf['events'], errors='coerce')

# map beambreaks to cm
conds = [
    (num < 4),
    (num < 8),
    (num < 16),
    (num < 32),
    (num < 64),
    (num < 128),
    (num < 196),
]
vals = [-1, 0, 5, 10, 20, 30, 40]
mapped = np.select(conds, vals, default=50)

# overwrite only numeric entries; strings stay as-is
mask = num.notna()
mdf.loc[mask, 'events'] = mapped[mask]

mdf['session_n'] = (mdf['session'].astype(str).str[-1].astype(int)-1)*10
mdf['trial_full'] = mdf['session_n'] + mdf['trial']

In [ ]:
successdf = mdf.groupby(['animal', 'trial', 'session', 'lesion', 'session_type', 'rewarded'], as_index=False).head(1)[['animal', 'session_type', 'lesion', 'rewarded']]
successdf = successdf.groupby(['animal', 'session_type', 'lesion'], observed=True).agg("mean").reset_index()

successdf_lesion = successdf[(successdf['lesion']=='Lesion') & (successdf['session_type']=='Reward')].sort_values(['rewarded'], ascending = False)
successdf_saline = successdf[(successdf['lesion']=='Saline') & (successdf['session_type']=='Reward')].sort_values(['rewarded'], ascending = False)
successdf = mdf.groupby(['animal', 'trial', 'session', 'lesion', 'session_type', 'session_n', 'rewarded'], as_index=False).head(1)[['animal', 'session_type', 'session_n','lesion', 'rewarded', 'trial', 'trial_full']]

successdf = mdf.groupby(['animal', 'trial', 'session', 'lesion', 'session_type', 'session_n', 'rewarded'], as_index=False).head(1)[['animal', 'session_type', 'session_n','lesion', 'rewarded', 'trial', 'trial_full']]

Figure 1C

In [ ]:
successdf = successdf.copy()
successdf["trial_full_true"] = successdf["trial"] + successdf["session_n"]
successdf = successdf[successdf["trial_full_true"] > 0]



summary = (
    successdf.groupby(["trial_full_true", "lesion"], observed=True)
             .agg(mean_rewarded=("rewarded", "mean"),
                  sem_rewarded=("rewarded", sem))
             .reset_index()
)

# --- plotting ---
palette = {"Saline": "#4C78A8", "Lesion": "#F58518"}

plt.figure(figsize=(7,4.2))

for les in ["Saline", "Lesion"]:
    sub = summary[summary["lesion"] == les].sort_values("trial_full_true")
    if sub.empty:
        continue
    x  = sub["trial_full_true"].to_numpy()
    mu = sub["mean_rewarded"].to_numpy()
    se = sub["sem_rewarded"].to_numpy()

    plt.plot(x, mu, lw=2, label=les, color=palette.get(les, "black"))
    plt.fill_between(x, mu - se, mu + se, alpha=0.25, color=palette.get(les, "gray"))

plt.xlabel("Trial (aligned index)")
plt.ylabel("Fraction Rewarded ± SEM")
plt.ylim(0, 1.05)
plt.xlim(summary["trial_full_true"].min(), summary["trial_full_true"].max())
plt.tight_layout()

# save (optional)
plt.savefig(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\Training_LesionSuccessRateTimecourse.svg')
plt.show()

In [ ]:
%%R -i successdf
library(lme4)
library(lmerTest)

successdf$animal<-as.factor(successdf$animal)
successdf$trial<-as.factor(successdf$trial)
successdf$session_n<-as.factor(successdf$session_n)


model <-glmer(rewarded ~ lesion * trial_full_true + (1|animal) + (1|session_n) + (1|trial), data= successdf, family=binomial)

summary(model)


Figure 1D

In [ ]:
successdf['prev_rewarded'] = successdf.groupby(['animal'])['rewarded'].shift(1)
successdf['CumulativeReward'] = successdf.groupby('animal')['rewarded'].cumsum()
trials_to_reward = successdf[successdf['CumulativeReward']==0].groupby(['animal', 'lesion']).size().reset_index(name = 'Trials')

In [ ]:
%%R -i trials_to_reward

t.test(Trials ~ lesion, data = trials_to_reward)

In [ ]:
%%R -i successdf -o emm

successdf <- subset(successdf, CumulativeReward > 0)

successdf$animal<-as.factor(successdf$animal)
successdf$trial<-as.factor(successdf$trial)
successdf$session_n<-as.factor(successdf$session_n)

model <- glmer(rewarded ~ lesion + (1|animal) + (1|session_n) + (1|trial), data= successdf, family=binomial)

emm <- emmeans(model, ~ lesion, type = "response")
print(pairs(emm, adjust = "bonferroni")) 
emm <- as.data.frame(emm)


In [ ]:
%%R -i successdf -o emm

successdf <- subset(successdf, CumulativeReward > 0)

successdf <- na.omit(successdf)

successdf$animal<-as.factor(successdf$animal)
successdf$trial<-as.factor(successdf$trial)
successdf$session_n<-as.factor(successdf$session_n)
successdf$prev_rewarded<-as.factor(successdf$prev_rewarded)


model <- glmer(rewarded ~ lesion * prev_rewarded + (1|animal) + (1|session_n) + (1|trial), data= successdf, family=binomial)

emm <- emmeans(model, ~ lesion|prev_rewarded, type = "response")
print(pairs(emm, adjust = "bonferroni"))  # lesion differences within session_type
emm <- as.data.frame(emm)
print(pairs(emmeans(model, ~ lesion, type = "response"), adjust = "bonferroni"))

Figure 1E

In [ ]:
tmp = mdf.copy()
tmp["event_num"] = pd.to_numeric(tmp["events"], errors="coerce")

# 2) per-trial max of numeric events
trial_max = (
    tmp.groupby(['animal', 'trial', 'session', 'lesion', 'session_type', 'rewarded',"session_n", "trial_full"], observed=True)["event_num"]
       .max()
       .reset_index(name="trial_max_event")
)

trial_max['reached_end'] = np.where(trial_max['trial_max_event']>=40, 1, 0)
trial_max['quit'] = abs(trial_max['reached_end']-1)

In [ ]:
trial_max = trial_max.copy()
trial_max["trial_full_true"] = trial_max["trial"] + trial_max["session_n"]
trial_max = trial_max[trial_max["trial_full_true"] > 0]


summary = (
    trial_max.groupby(["trial_full_true", "lesion"], observed=True)
             .agg(mean_rewarded=("trial_max_event", "mean"),
                  sem_rewarded=("trial_max_event", sem))
             .reset_index()
)

palette = {"Saline": "#4C78A8", "Lesion": "#F58518"}

plt.figure(figsize=(7,4.2))

for les in ["Saline", "Lesion"]:
    sub = summary[summary["lesion"] == les].sort_values("trial_full_true")
    if sub.empty:
        continue
    x  = sub["trial_full_true"].to_numpy()
    mu = sub["mean_rewarded"].to_numpy()
    se = sub["sem_rewarded"].to_numpy()

    plt.plot(x, mu, lw=2, label=les, color=palette.get(les, "black"))
    plt.fill_between(x, mu - se, mu + se, alpha=0.25, color=palette.get(les, "gray"))

plt.xlabel("Trial (aligned index)")
plt.ylabel("Furthest Distance Attained")
plt.ylim(0, 45)
plt.xlim(summary["trial_full_true"].min(), summary["trial_full_true"].max())
plt.tight_layout()

# save (optional)
plt.savefig(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\Training_LesionDistanceTimecourse.svg')
plt.show()

In [ ]:
%%R -i trial_max
library(lme4)
library(lmerTest)

trial_max$animal<-as.factor(trial_max$animal)
trial_max$trial<-as.factor(trial_max$trial)
trial_max$session_n<-as.factor(trial_max$session_n)


model <-lmer(trial_max_event ~ lesion * trial_full_true + (1|animal) +  (1|trial), data= trial_max)

print(anova(model))

Figure 1F

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LinearSegmentedColormap, ListedColormap

lesion_order  = ["Saline", "Lesion"]
session_order = ["Reward", "Monster"]
vmin, vmax    = 0, 40

base_min_color = "#068067"
base_max_color = "#cbde6b"
reward_color   = "#f9f06d"

cbar_label    = "Max Distance (cm)"
figsize       = (10, 7.5)
savepath      = r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\Training_RewardLesionDistanceHeatmap.svg'

row_order_by_lesion = {
    "Saline": successdf_saline["animal"].drop_duplicates().tolist() if "successdf_saline" in globals() else [],
    "Lesion": successdf_lesion["animal"].drop_duplicates().tolist() if "successdf_lesion" in globals() else [],
}

df = trial_max.copy()
df["lesion"]       = pd.Categorical(df["lesion"], categories=lesion_order, ordered=True)
df["session_type"] = pd.Categorical(df["session_type"], categories=session_order, ordered=True)

df["trial_max_event"] = pd.to_numeric(df["trial_max_event"], errors="coerce")
df["rewarded"]        = pd.to_numeric(df["rewarded"], errors="coerce").fillna(0).astype(int)  # expects a rewarded column

all_trials = np.sort(df["trial_full"].unique())

row_animals = {les: df.loc[df["lesion"] == les, "animal"].nunique() for les in lesion_order}
row_heights = [max(row_animals.get(les, 1), 1) for les in lesion_order]

base_cmap   = LinearSegmentedColormap.from_list("base_cmap", [base_min_color, base_max_color])
reward_cmap = ListedColormap([reward_color])

fig = plt.figure(figsize=figsize)
gs  = fig.add_gridspec(
    nrows=2, ncols=2,
    height_ratios=row_heights,
    wspace=0.05, hspace=0.08
)

axes = {}
for r, les in enumerate(lesion_order):
    for c, ses in enumerate(session_order):
        ax = fig.add_subplot(gs[r, c])
        axes[(les, ses)] = ax

        sub = df[(df["lesion"] == les) & (df["session_type"] == ses)]

        mat = (
            sub.pivot_table(
                index="animal", columns="trial_full",
                values="trial_max_event", aggfunc="mean"
            )
            .reindex(columns=all_trials)
        )

        reward_mat = (
            sub.pivot_table(
                index="animal", columns="trial_full",
                values="rewarded", aggfunc="max"
            )
            .reindex(columns=all_trials)
        )

        desired_animals = [a for a in row_order_by_lesion.get(les, []) if a in mat.index]
        if len(desired_animals):
            mat = mat.reindex(index=desired_animals)
            reward_mat = reward_mat.reindex(index=desired_animals)
        else:
            mat = mat.sort_index()
            reward_mat = reward_mat.reindex(index=mat.index)

        sns.heatmap(
            mat,
            ax=ax,
            cmap=base_cmap,
            vmin=vmin, vmax=vmax,
            cbar=False,
            linewidths=0, linecolor=None,
            xticklabels=True, yticklabels=True
        )

        reward_bool = (reward_mat.fillna(0).astype(int) == 1)

        sns.heatmap(
            reward_mat,
            ax=ax,
            cmap=reward_cmap,
            vmin=0, vmax=1,
            cbar=False,
            linewidths=0, linecolor=None,
            mask=~reward_bool,
            xticklabels=False, yticklabels=False
        )

        ax.set_xlabel("Trial" if r == 1 else "")
        ax.set_ylabel("Animal" if c == 0 else "")
        ax.tick_params(axis="x", labelrotation=0)

        if r == 0:
            ax.set_title(ses, pad=6, fontsize=12)
        if c == 1:
            ax.text(1.02, 0.5, les, transform=ax.transAxes, rotation=90,
                    va="center", ha="left", fontsize=11)

        for s in ax.spines.values():
            s.set_visible(False)

norm = Normalize(vmin=vmin, vmax=vmax)
sm   = plt.cm.ScalarMappable(cmap=base_cmap, norm=norm)
sm.set_array([])
cax = fig.add_axes([0.92, 0.20, 0.02, 0.6])
fig.colorbar(sm, cax=cax, label=cbar_label)

plt.subplots_adjust(left=0.08, right=0.90, top=0.92, bottom=0.08)
plt.savefig(savepath)
plt.show()

Figure 1G

%%R -i trial_max 
library(survival)
library(coxme)

m_surv_me <- coxme(
  Surv(time = trial_max$trial_max_event, event = trial_max$quit) ~ 
    lesion + (1 | animal) + (1|session_n) + (1|trial),
  data = trial_max
)

summary(m_surv_me)

In [ ]:
%%R -i trial_max

S <- Surv(time = trial_max$trial_max_event, event = trial_max$quit)

# Kaplan–Meier curves
fit_km <- survfit(S ~ lesion, data = trial_max)

# 1) Log-rank test (KM curve difference)
lr <- survdiff(S ~ lesion, data = trial_max)
print(lr)

logp <- pchisq(lr$chisq,
               df = length(lr$n) - 1,
               lower.tail = FALSE,
               log.p = TRUE)

exp10 <- floor(logp / log(10))
mant  <- exp(logp - exp10 * log(10))

cat("p =", sprintf("%.1fe%d", mant, exp10), "\n")

In [ ]:
%%R -i trial_max

library(survival)
library(survminer)
library(svglite)

S <- Surv(time = trial_max$trial_max_event, event = trial_max$quit)

# Kaplan–Meier fit by lesion group
fit_km <- survfit(S ~ lesion, data = trial_max)

Plot 
km_plot <- ggsurvplot(
  fit_km,
  data = trial_max,
  conf.int = TRUE,               # show 95% CIs
  risk.table = TRUE,             # show # at risk below plot
  pval = TRUE,                   # log-rank test
  xlab = "Distance (cm)",
  ylab = "Probability of continuing approach",
  legend.title = "Group",
  legend.labs = levels(trial_max$lesion),
  ggtheme = theme_classic(),
  palette = c("#1B9E77", "#D95F02"),  # optional custom colors
  xlim = c(0, 40)
)

km_plot

In [ ]:
%%R -i trial_max 

trial_max$lesion_num      <- ifelse(trial_max$lesion %in% c("Lesion","lesion"), 1, 0)
trial_max$quit            <- as.integer(trial_max$quit)       # 1 = failed early, 0 = rewarded
trial_max$session_n       <- factor(trial_max$session_n)
trial_max$trial_max_event <- as.numeric(trial_max$trial_max_event)
trial_max$animal          <- factor(trial_max$animal)

m_tvc_frail <- coxph(
  Surv(trial_max_event, quit) ~ lesion_num + tt(lesion_num) + strata(session_n) + frailty(animal),
  data = trial_max,
  tt = function(x, t, ...) as.numeric(x) * log(pmax(t, 1e-6) + 1)
)
summary(m_tvc_frail)

In [ ]:
%%R -i trial_max 

# Prep
trial_max$lesion_num      <- ifelse(trial_max$lesion %in% c("Lesion","lesion"), 1, 0)
trial_max$quit            <- as.integer(trial_max$quit)       # 1 = failed early, 0 = rewarded
trial_max$session_n       <- factor(trial_max$session_n)
trial_max$trial_max_event <- as.numeric(trial_max$trial_max_event)
trial_max$animal          <- factor(trial_max$animal)

# Cox PH with time-varying lesion effect + gamma frailty for animal
m_tvc_frail <- coxph(
  Surv(trial_max_event, quit) ~ lesion_num + tt(lesion_num) + strata(session_n) + frailty(animal),
  data = trial_max,
  tt = function(x, t, ...) as.numeric(x) * log(pmax(t, 1e-6) + 1)
)
summary(m_tvc_frail)

# Extract coefs / vcov
co <- coef(m_tvc_frail)
b1 <- co["lesion_num"]
b2 <- co["tt(lesion_num)"]

dist_seq <- seq(1, 40, by = 0.5)
logHR <- b1 + b2 * log(dist_seq + 1)
HR    <- exp(logHR)

vc <- vcov(m_tvc_frail)
var_logHR <- vc["lesion_num","lesion_num"] +
             (log(dist_seq + 1))^2 * vc["tt(lesion_num)","tt(lesion_num)"] +
             2 * log(dist_seq + 1) * vc["lesion_num","tt(lesion_num)"]
SE <- sqrt(pmax(var_logHR, 0))
HR_low  <- exp(logHR - 1.96*SE)
HR_high <- exp(logHR + 1.96*SE)
signif  <- (HR_low > 1 | HR_high < 1)

df_HR <- data.frame(dist = dist_seq, HR, HR_low, HR_high, signif)

km_ratio <- ggplot(df_HR, aes(x = dist, y = HR)) +
  geom_ribbon(aes(ymin = HR_low, ymax = HR_high, fill = signif), alpha = 0.3) +
  geom_line(size = 1.2) +
  geom_hline(yintercept = 1, linetype = "dashed") +
  scale_fill_manual(values = c("TRUE" = "skyblue", "FALSE" = "grey80"),
                    labels = c("ns","p < .05"), name = "Significance") +
  labs(x = "Distance (cm)", y = "Hazard ratio (Lesion / Control)",
       title = "Distance-dependenteffect ") +
  theme_classic(base_size = 14)

km_ratio

Figure 1H

In [ ]:
trial_max['prev_rewarded'] = trial_max.groupby(['animal'])['rewarded'].shift(1)
trial_max['CumulativeReward'] = trial_max.groupby('animal')['rewarded'].cumsum()
trial_max['FirstReward'] = np.where(trial_max['CumulativeReward']==0, 'before', 'after')

In [ ]:
%%R -i trial_max 

trial_max <- subset(trial_max, trial_max_event>25)
trial_max$session_n <- as.factor(trial_max$session_n)


model <- glmer(rewarded  ~ lesion*session_n + (1|animal) + (1|trial), data= trial_max, family=binomial)

emm <- emmeans(model, ~ lesion, type = "response")
print(pairs(emm, adjust = "bonferroni")) 

print(emm)

print(anova(model))
print(car::Anova(model, type = 3))

In [ ]:
g_successdf = trial_max[trial_max['trial_max_event']>25].copy()

g_successdf = g_successdf.groupby(["session_n", "animal", "lesion"], observed=True).agg(mean_rewarded=("rewarded", "mean"),sem_rewarded=("rewarded", sem)).reset_index()
summary = g_successdf.groupby(["session_n", "lesion"], observed=True).agg(mean_rewarded=("mean_rewarded", "mean"),sem_rewarded=("mean_rewarded", sem)).reset_index()

summary['session_n'] = (summary['session_n']/10) + 1
 
# --- plotting ---
palette = {"Saline": "#4C78A8", "Lesion": "#F58518"}

plt.figure(figsize=(7,4.2))

for les in ["Saline", "Lesion"]:
    sub = summary[summary["lesion"] == les].sort_values("session_n")
    if sub.empty:
        continue
    x  = sub["session_n"].to_numpy()
    mu = sub["mean_rewarded"].to_numpy()
    se = sub["sem_rewarded"].to_numpy()

    plt.plot(x, mu, lw=2, label=les, color=palette.get(les, "black"))
    plt.fill_between(x, mu - se, mu + se, alpha=0.25, color=palette.get(les, "gray"))

plt.xlabel("Trial (aligned index)")
plt.ylabel("Fraction Rewarded ± SEM")
plt.ylim(0, 1.05)
plt.xlim(summary["session_n"].min(), summary["session_n"].max())
plt.tight_layout()

# save (optional)
plt.savefig(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\Training_LesionSuccessRateTimecourse_Proximal.svg')
plt.show()

In [ ]:
g_successdf = trial_max.copy()

g_successdf = g_successdf.groupby(["session_n", "animal", "lesion"], observed=True).agg(mean_rewarded=("rewarded", "mean"),sem_rewarded=("rewarded", sem)).reset_index()
summary = g_successdf.groupby(["session_n", "lesion"], observed=True).agg(mean_rewarded=("mean_rewarded", "mean"),sem_rewarded=("mean_rewarded", sem)).reset_index()

summary['session_n'] = (summary['session_n']/10) + 1
 
# --- plotting ---
palette = {"Saline": "#4C78A8", "Lesion": "#F58518"}

plt.figure(figsize=(7,4.2))

for les in ["Saline", "Lesion"]:
    sub = summary[summary["lesion"] == les].sort_values("session_n")
    if sub.empty:
        continue
    x  = sub["session_n"].to_numpy()
    mu = sub["mean_rewarded"].to_numpy()
    se = sub["sem_rewarded"].to_numpy()

    plt.plot(x, mu, lw=2, label=les, color=palette.get(les, "black"))
    plt.fill_between(x, mu - se, mu + se, alpha=0.25, color=palette.get(les, "gray"))

plt.xlabel("Trial (aligned index)")
plt.ylabel("Fraction Rewarded ± SEM")
plt.ylim(0, 1.05)
plt.xlim(summary["session_n"].min(), summary["session_n"].max())
plt.tight_layout()

# save (optional)
plt.savefig(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\Training_LesionSuccessRateTimecourse_Distal.svg')
plt.show()

In [ ]:
trial_max['MadeTo25'] = np.where(trial_max['trial_max_event']>=25, 1, 0)
g_successdf = trial_max.copy()
g_successdf = g_successdf.groupby(["session_n", "animal", "lesion"], observed=True).agg(mean_rewarded=("MadeTo25", "mean"),sem_rewarded=("MadeTo25", sem)).reset_index()
summary = g_successdf.groupby(["session_n", "lesion"], observed=True).agg(mean_rewarded=("mean_rewarded", "mean"),sem_rewarded=("mean_rewarded", sem)).reset_index()

summary['session_n'] = (summary['session_n']/10) + 1
 
# --- plotting ---
palette = {"Saline": "#4C78A8", "Lesion": "#F58518"}

plt.figure(figsize=(7,4.2))

for les in ["Saline", "Lesion"]:
    sub = summary[summary["lesion"] == les].sort_values("session_n")
    if sub.empty:
        continue
    x  = sub["session_n"].to_numpy()
    mu = sub["mean_rewarded"].to_numpy()
    se = sub["sem_rewarded"].to_numpy()

    plt.plot(x, mu, lw=2, label=les, color=palette.get(les, "black"))
    plt.fill_between(x, mu - se, mu + se, alpha=0.25, color=palette.get(les, "gray"))

plt.xlabel("Session (aligned index)")
plt.ylabel("Probability of getting to 25cm ± SEM")
plt.ylim(0, 1.05)
plt.xlim(summary["session_n"].min(), summary["session_n"].max())
plt.tight_layout()

# save (optional)
plt.savefig(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\Training_LesionSuccessRateTimecourse_PGetto25.svg')
plt.show()

In [ ]:
%%R -i trial_max 
library(lme4)
library(emmeans)
library(car)


model <- glmer(MadeTo25  ~ lesion*session_n + (1|animal) + (1|trial), data= trial_max, family=binomial)
emm <- emmeans(model, ~ lesion, type = "response")
print(pairs(emm, adjust = "bonferroni")) 

print(emm)

print(car::Anova(model, type = 3))

print(anova(model))

print(summary(model))